In [5]:
import os, glob
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import pymc as pm
import pytensor.tensor as pt
import arviz as az

import traceback
import pickle

# -----------------------
# Config
# -----------------------
NETID = "k16v981"

BASIN = "gulf_oman"   # change per run: arabian_gulf, gulf_oman, red_sea

BASIN_NC = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/basin_anoms/era5_sst_anom_{BASIN}_1950_2025.nc"
IDX_CSV  = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/roni_dmi_monthly_1950_2025.csv"

# output
OUT_IDATA = f"/home/{NETID}/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/gpd_{BASIN}_roni_dmi_idata.nc"

RANDOM_SEED = 72

# predictor lags (months)
ENSO_LAG = 2   # RONI lag
IOD_LAG  = 1   # DMI lag

# POT choices
Q = 0.95          # threshold quantile per grid cell
MIN_EVENTS = 5    # drop grid cells with too-few exceedances (stability)
XI_LOWER = -0.3   # optional bounds for xi (tail shape)
XI_UPPER = 0.5

# -----------------------
# Load basin SST anomalies
# -----------------------
ds = xr.open_dataset(BASIN_NC)
da = ds["sst_anom"]  # (time, lat, lon) may be sub-daily or monthly

lat_name = "latitude" if "latitude" in da.coords else "lat"
lon_name = "longitude" if "longitude" in da.coords else "lon"

# shift lon to [-180, 180] if needed for mapping
if float(da[lon_name].max()) > 180:
    lon = da[lon_name]
    lon_new = ((lon + 180) % 360) - 180
    da = da.assign_coords({lon_name: lon_new}).sortby(lon_name)

# ---- HARD GUARD / FIX: ensure SST time is monthly month-start ----
t0 = pd.DatetimeIndex(pd.to_datetime(da["time"].values))
is_month_start_midnight = (t0.day == 1).all() and (t0.hour == 0).all() and (t0.minute == 0).all()

if not is_month_start_midnight:
    print("⚠️ SST time is not monthly. Resampling to monthly month-start (MS) means...")
    da = da.resample(time="MS").mean(skipna=True)

# stack spatial dims -> (time, space)
da_st = da.stack(space=(lat_name, lon_name))

# keep only wet basin points: non-NaN at first time
valid_space = np.isfinite(da_st.isel(time=0).values)
da_st = da_st.isel(space=valid_space)

Y = da_st.values.astype("float32")  # (T, S) monthly anomalies
time = pd.to_datetime(da_st["time"].values)

# -----------------------
# Restrict to warm season (JJAS) BEFORE aligning indices / POT
# -----------------------
jjas_mask = pd.DatetimeIndex(time).month.isin([6, 7, 8, 9])

# Apply to the stacked DataArray so coordinates stay consistent
da_st = da_st.isel(time=jjas_mask)

# Re-extract arrays after subsetting
Y = da_st.values.astype("float32")
time = pd.to_datetime(da_st["time"].values)

# Update shapes
T, S = Y.shape

print(f"✅ Restricted to JJAS: T={T} months, S={S} wet points")

# spatial coords list (S,)
space_index = da_st["space"].to_index()
lats = np.array([x[0] for x in space_index], dtype="float32")
lons = np.array([x[1] for x in space_index], dtype="float32")

X_space = np.column_stack([lons, lats]).astype("float32")  # (S,2) [lon,lat]
T, S = Y.shape

ds.close()

# -----------------------
# Load ENSO / IOD indices
# -----------------------
idx = pd.read_csv(IDX_CSV)

# Parse time
if "time" in idx.columns:
    idx["time"] = pd.to_datetime(idx["time"])
elif {"year", "month"}.issubset(idx.columns):
    idx["time"] = pd.to_datetime(dict(year=idx["year"], month=idx["month"], day=1))
else:
    raise ValueError(
        f"Index CSV needs either 'time' or ('year','month') columns. Found: {idx.columns.tolist()}"
    )

def pick_col(cols, key):
    cols_l = {c.lower(): c for c in cols}
    for cl, orig in cols_l.items():
        if cl == key or key in cl:
            return orig
    return None

roni_col = pick_col(idx.columns, "roni")
dmi_col  = pick_col(idx.columns, "dmi")
if roni_col is None or dmi_col is None:
    raise ValueError(f"Could not identify RONI/DMI columns. Columns: {idx.columns.tolist()}")

idx = idx.set_index("time").sort_index()

# Deduplicate index times if needed
if idx.index.duplicated().any():
    print("⚠️ Duplicate times found in index CSV — deduplicating (keep last).")
    idx = idx[~idx.index.duplicated(keep="last")]

# Align indices to SST monthly time
if pd.Index(time).duplicated().any():
    raise ValueError("SST monthly time axis contains duplicates (unexpected).")


# Extract lagged predictors (monthly)
# For SST month t:
#   RONI predictor = RONI(t - ENSO_LAG)
#   DMI  predictor = DMI(t - IOD_LAG)
N_series = idx[roni_col].shift(ENSO_LAG)
D_series = idx[dmi_col].shift(IOD_LAG)

idx_lagged = pd.DataFrame({
    "N_lag": N_series,
    "D_lag": D_series,
}).sort_index()

idx_aligned = idx_lagged.reindex(time)

if idx_aligned[["N_lag", "D_lag"]].isna().any().any():
    missing = idx_aligned[idx_aligned["N_lag"].isna() | idx_aligned["D_lag"].isna()]
    raise ValueError(
        f"Missing lagged index values after aligning to SST months.\n"
        f"First missing rows:\n{missing.head()}\n"
        f"Index range: {idx.index.min()} → {idx.index.max()}\n"
        f"SST   range: {time.min()} → {time.max()}\n"
        f"ENSO_LAG={ENSO_LAG}, IOD_LAG={IOD_LAG}"
    )

N = idx_aligned["N_lag"].astype("float32").values
D = idx_aligned["D_lag"].astype("float32").values

# standardize after lagging/alignment
N = (N - N.mean()) / N.std()
D = (D - D.mean()) / D.std()
ND = (N * D).astype("float32")

print(f"✅ Data aligned: T={T} months, S={S} wet points")
print(f"✅ Using lagged predictors: RONI lag={ENSO_LAG} month(s), DMI lag={IOD_LAG} month(s)")

# -----------------------
# Build km coords for space (for mapping / optional spatial priors later)
# -----------------------
lon = X_space[:, 0].astype("float64")
lat = X_space[:, 1].astype("float64")

lat0 = float(lat.mean())
km_per_deg_lat = 111.32
km_per_deg_lon = 111.32 * np.cos(np.deg2rad(lat0))

x_km = (lon - float(lon.mean())) * km_per_deg_lon
y_km = (lat - float(lat.mean())) * km_per_deg_lat
X_km = np.column_stack([x_km, y_km]).astype("float32")

# -----------------------
# POT / GPD exceedance extraction (flattened "event" table)
# -----------------------
u = np.nanquantile(Y, Q, axis=0).astype("float32")     # threshold per cell, (S,)
exc = Y > u[None, :]                                   # (T,S)
t_idx, s_idx = np.where(exc)

z = (Y[t_idx, s_idx] - u[s_idx]).astype("float32")     # exceedances, (E,)
N_e  = N[t_idx].astype("float32")
D_e  = D[t_idx].astype("float32")
ND_e = ND[t_idx].astype("float32")

# stability: drop cells with too-few exceedances, and filter events accordingly
counts = np.bincount(s_idx, minlength=S)
keep_space = counts >= MIN_EVENTS
if keep_space.sum() < 5:
    raise RuntimeError(f"Too few grid cells with >= {MIN_EVENTS} exceedances. Try lower MIN_EVENTS or lower Q.")

keep_event = keep_space[s_idx]
z   = z[keep_event]
N_e = N_e[keep_event]
D_e = D_e[keep_event]
ND_e= ND_e[keep_event]
s_e = s_idx[keep_event].astype("int32")

# remap space indices to 0..S_keep-1
old_to_new = -np.ones(S, dtype="int32")
old_to_new[np.where(keep_space)[0]] = np.arange(keep_space.sum(), dtype="int32")
s_e = old_to_new[s_e]

# subset coord arrays to kept spaces (for later plotting)
lats_k = lats[keep_space]
lons_k = lons[keep_space]
X_km_k = X_km[keep_space]
u_k    = u[keep_space]

E = z.size
S_k = keep_space.sum()

print(f"✅ POT built: Q={Q}, kept S={S_k} cells, E={E} exceedances (avg {E/S_k:.1f} per cell)")

# -----------------------
# GPD model: nonstationary scale via ENSO/IOD, hierarchical cell effects
# (fast + directly focused on extremes)
# -----------------------
coords = {
    "event": np.arange(E),
    "space": np.arange(S_k),
    "xy": ["x_km", "y_km"],
}

def gpd_logp(z, sigma, xi, eps=1e-12, xi_tol=1e-6):
    sigma = sigma + eps
    t = 1 + xi * z / sigma

    logp_gpd = -pt.log(sigma) - (1 + 1/xi) * pt.log(t)
    logp_exp = -pt.log(sigma) - z / sigma  # xi -> 0 limit

    logp_vec = pt.switch(pt.abs(xi) < xi_tol, logp_exp, logp_gpd)
    logp_vec = pt.switch(t > 0, logp_vec, -np.inf)
    return pt.sum(logp_vec)

with pm.Model(coords=coords) as model:
    z_obs = pm.ConstantData("z", z, dims="event")
    N_t   = pm.MutableData("N_t",  N_e,  dims="event")
    D_t   = pm.MutableData("D_t",  D_e,  dims="event")
    ND_t  = pm.MutableData("ND_t", ND_e, dims="event")
    s_id  = pm.ConstantData("s_id", s_e.astype("int32"), dims="event")

    xi = pm.TruncatedNormal("xi", mu=0.05, sigma=0.15, lower=XI_LOWER, upper=XI_UPPER)

    a_bar   = pm.Normal("a_bar",   0.0, 1.0)
    bN_bar  = pm.Normal("bN_bar",  0.0, 0.5)
    bD_bar  = pm.Normal("bD_bar",  0.0, 0.5)
    bND_bar = pm.Normal("bND_bar", 0.0, 0.5)

    a_sd   = pm.HalfNormal("a_sd",   0.8)
    bN_sd  = pm.HalfNormal("bN_sd",  0.3)
    bD_sd  = pm.HalfNormal("bD_sd",  0.3)
    bND_sd = pm.HalfNormal("bND_sd", 0.2)

    # non-centered
    a_z   = pm.Normal("a_z", 0, 1, dims="space")
    bN_z  = pm.Normal("bN_z", 0, 1, dims="space")
    bD_z  = pm.Normal("bD_z", 0, 1, dims="space")
    bND_z = pm.Normal("bND_z", 0, 1, dims="space")

    a_s   = pm.Deterministic("a_s",   a_bar   + a_sd   * a_z,   dims="space")
    bN_s  = pm.Deterministic("bN_s",  bN_bar  + bN_sd  * bN_z,  dims="space")
    bD_s  = pm.Deterministic("bD_s",  bD_bar  + bD_sd  * bD_z,  dims="space")
    bND_s = pm.Deterministic("bND_s", bND_bar + bND_sd * bND_z, dims="space")

    log_sigma = a_s[s_id] + bN_s[s_id]*N_t + bD_s[s_id]*D_t + bND_s[s_id]*ND_t
    sigma = pm.Deterministic("sigma", 1e-6 + pt.exp(log_sigma), dims="event")

    pm.DensityDist(
        "z_like",
        sigma, xi,                              # <-- positional dist_params
        logp=lambda z, sigma, xi: gpd_logp(z, sigma, xi),
        observed=z_obs,
        dims="event",
    )

    idata = pm.sample(
        draws=2000,
        tune=2000,
        chains=4,
        cores=4,
        target_accept=0.97,
        max_treedepth=15,
        random_seed=RANDOM_SEED,
    )


def safe_save_idata(idata, out_path):
    """
    Try multiple save strategies so we don't lose a long run.
    """
    try:
        az.to_netcdf(idata, out_path)
        print(f"✅ ArviZ NetCDF saved: {out_path}")
        return
    except Exception as e:
        print("⚠️ az.to_netcdf failed. Reason:")
        print(e)
        traceback.print_exc()

    # ---- fallback 1: save posterior only ----
    try:
        post_path = out_path.replace(".nc", "_posterior.nc")
        az.to_netcdf(idata.posterior, post_path)
        print(f"✅ Fallback: posterior-only saved to {post_path}")
    except Exception as e:
        print("⚠️ Posterior-only save failed.")
        print(e)

    # ---- fallback 2: pickle full idata ----
    try:
        pkl_path = out_path.replace(".nc", ".pkl")
        with open(pkl_path, "wb") as f:
            pickle.dump(idata, f, protocol=pickle.HIGHEST_PROTOCOL)
        print(f"✅ Fallback: pickled idata saved to {pkl_path}")
    except Exception as e:
        print("❌ Pickle save also failed.")
        print(e)
        print("🚨 WARNING: idata exists in memory only!")
        
safe_save_idata(idata, OUT_IDATA)

⚠️ SST time is not monthly. Resampling to monthly month-start (MS) means...
✅ Restricted to JJAS: T=304 months, S=151 wet points
⚠️ Duplicate times found in index CSV — deduplicating (keep last).
✅ Data aligned: T=304 months, S=151 wet points
✅ Using lagged predictors: RONI lag=2 month(s), DMI lag=1 month(s)
✅ POT built: Q=0.95, kept S=151 cells, E=2416 exceedances (avg 16.0 per cell)


Auto-assigning NUTS sampler...
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [xi, a_bar, bN_bar, bD_bar, bND_bar, a_sd, bN_sd, bD_sd, bND_sd, a_z, bN_z, bD_z, bND_z]


Sampling 4 chains for 2_000 tune and 2_000 draw iterations (8_000 + 8_000 draws total) took 70 seconds.


✅ ArviZ NetCDF saved: /home/k16v981/my_work/code/arabian_peninsula/bayesian_extremes/data/sst/gpd_gulf_oman_roni_dmi_idata.nc


In [6]:
# core diagnostics
print(az.summary(
    idata,
    var_names=["xi", "a_bar", "a_sd", "bN_bar", "bN_sd", "bD_bar", "bD_sd", "bND_bar", "bND_sd"],
    round_to=3
))

# divergences?
div = int(idata.sample_stats["diverging"].sum().values)
print("Divergences:", div)

# max tree depth hits?
if "tree_depth" in idata.sample_stats:
    td = idata.sample_stats["tree_depth"].values
    print("Tree depth max:", td.max(), " / mean:", td.mean())

          mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd   ess_bulk  \
xi      -0.071  0.025  -0.119   -0.026      0.000      0.0   4554.054   
a_bar   -1.487  0.034  -1.551   -1.423      0.000      0.0   5760.496   
a_sd     0.167  0.029   0.112    0.221      0.001      0.0   2426.561   
bN_bar  -0.181  0.021  -0.220   -0.142      0.000      0.0  11093.852   
bN_sd    0.028  0.021   0.000    0.065      0.000      0.0   3512.074   
bD_bar   0.314  0.024   0.269    0.360      0.000      0.0  10862.770   
bD_sd    0.031  0.023   0.000    0.073      0.000      0.0   3688.688   
bND_bar -0.020  0.024  -0.062    0.026      0.000      0.0   9754.496   
bND_sd   0.037  0.027   0.000    0.084      0.000      0.0   2559.257   

         ess_tail  r_hat  
xi       5859.543  1.001  
a_bar    5961.477  1.001  
a_sd     2685.109  1.002  
bN_bar   6832.027  1.001  
bN_sd    3759.946  1.001  
bD_bar   6520.963  1.000  
bD_sd    4035.861  1.000  
bND_bar  6636.208  1.000  
bND_sd   3371.111  1.001  